# Neural WFA Inversion (Time Series Simulation)

This script demonstrates the usage of the refactored `neural_wfa` package for
inverting solar spectropolarimetric data using Neural Fields on a **simulated time series**.
The time series is generated by amplifying the Stokes Q, U, V signals of a single snapshot.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import astropy.io.fits as fits
import os
import sys

# Ensure src is in path if running locally
sys.path.append("src")
sys.path.append("../src")

from neural_wfa import Observation, WFAProblem
from neural_wfa.physics import LineInfo
from neural_wfa.nn import MLP
from neural_wfa.optimization import NeuralSolver
from neural_wfa.utils.viz import set_params
from neural_wfa.utils.viz import plot_wfa_results, torch2numpy

set_params()

## 1. Load Data & Simulate Time Series

In [ ]:
datadir = "example_py/plage_sst/"
if not os.path.exists(datadir):
    datadir = "plage_sst/"

# Original Single Snapshot (Ny, Nx, 4, Nw)
img_raw = np.ascontiguousarray(
    fits.open(datadir + "CRISP_5173_plage_dat.fits", "readonly")[0].data,
    dtype="float32",
)
xl = np.ascontiguousarray(
    fits.open(datadir + "CRISP_5173_plage_wav.fits", "readonly")[0].data,
    dtype="float32",
)

print("Original Data shape:", img_raw.shape)
ny_orig, nx_orig, ns, nw = img_raw.shape

# --- Simulate Time Series ---
Nt = 4
amplify_factor = 1.5

print(f"Generating {Nt} frames with Q/U/V amplification factor {amplify_factor}...")

img_series_list = []

for t in range(Nt):
    frame = img_raw.copy()

    # Scale factor for this timestep: 1.5^t
    scale = amplify_factor**t

    # Apply to Q, U, V (indices 1, 2, 3)
    frame[:, :, 1, :] *= scale
    frame[:, :, 2, :] *= scale
    frame[:, :, 3, :] *= scale

    img_series_list.append(frame)

# Stack to (Nt, Ny, Nx, Ns, Nw)
img_series = np.stack(img_series_list, axis=0)
print("Time Series Data shape:", img_series.shape)
nt, ny, nx, ns, nw = img_series.shape

## 2. Setup Problem

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Observation (5D input)
obs = Observation(img_series, xl, active_wav_idx=[5, 6, 7], device=str(device))
print(f"Observation loaded. Grid Shape: {obs.grid_shape}")

# Line Parameters
lin = LineInfo(5173)

# WFA Physics Engine
problem = WFAProblem(obs, lin, device=device)

## 3. Initialize Neural Fields (3D Coordinates: t, y, x)

In [ ]:
# Coordinate Grid (normalized -1 to 1 for all dimensions)
t_norm = np.linspace(-1, 1, nt)
y_norm = np.linspace(-1, 1, ny)
x_norm = np.linspace(-1, 1, nx)

# Create 3D meshgrid (t, y, x) -> shape (Nt, Ny, Nx, 3)
TT, YY, XX = np.meshgrid(t_norm, y_norm, x_norm, indexing="ij")
coords = np.stack([TT, YY, XX], axis=-1).reshape(-1, 3)  # Flatten to (Nt*Ny*Nx, 3)
coords = torch.from_numpy(coords.astype(np.float32)).to(device)

print(f"Coordinate Grid Shape: {coords.shape} (should be (Nt*Ny*Nx, 3))")

# Model for Blos (3D input now)
model_blos = MLP(
    dim_in=3,  # <--- Changed from 2 to 3 for time dimension
    dim_out=1,
    dim_hidden=64,
    num_resnet_blocks=2,
    fourier_features=True,
    m_freqs=512,
    sigma=40.0,  # Adjust sigma for 3D if needed
    tune_beta=False,
)

# Model for BQU (3D input now)
model_bqu = MLP(
    dim_in=3,  # <--- Changed from 2 to 3 for time dimension
    dim_out=2,
    dim_hidden=64,
    num_resnet_blocks=2,
    fourier_features=True,
    m_freqs=512,
    sigma=8.0,
    tune_beta=False,
)

## 4. Train using Neural Solver

In [ ]:
solver = NeuralSolver(
    problem=problem,
    model_blos=model_blos,
    model_bqu=model_bqu,
    coordinates=coords,
    lr=5e-4,
    batch_size=200000,
    device=device,
)
# Update normalization to match legacy neural script defaults
solver.set_normalization(w_blos=1.0, w_bqu=1000.0)

print("Training Phase 1: Blos Only...")
solver.train(n_epochs=400, optimize_blos=True, optimize_bqu=False)
loss_blos = np.array(solver.loss_history)
lr_blos = np.array(solver.lr_history)
solver.loss_history = []
solver.lr_history = []

print("Training Phase 2: BQU Only...")
solver.train(n_epochs=400, optimize_blos=False, optimize_bqu=True)
loss_bqu = np.array(solver.loss_history)
lr_bqu = np.array(solver.lr_history)


from neural_wfa.utils.viz import plot_loss

# Phase 1
plot_loss({"loss": loss_blos, "lr": lr_blos})
plt.savefig("ref_neural_time_loss_blos.png", dpi=300)
plt.close()

# Phase 2
plot_loss({"loss": loss_bqu, "lr": lr_bqu})
plt.savefig("ref_neural_time_loss_bqu.png", dpi=300)
plt.close()

## 5. Visualize Results & Analysis

In [ ]:
final_field = solver.get_full_field()

# 1. Magnetic Field Maps (Using new convenience properties)
# Shape should be (Nt, Ny, Nx) if grid_shape is correctly set
blos_map = torch2numpy(final_field.blos_map)
btrans_map = torch2numpy(final_field.btrans_map)
azi_map = torch2numpy(final_field.phi_map)

print(f"Output Shape: blos_map = {blos_map.shape} (Expected: ({nt}, {ny}, {nx}))")

# Plot LAST frame for comparison with explicit solver
if len(blos_map.shape) == 3:  # (Nt, Ny, Nx)
    plot_wfa_results(
        blos_map[-1],
        btrans_map[-1],
        azi_map[-1],
        save_name="ref_neural_time_results_last_frame.png",
    )
else:
    # Fallback if shape is flat (legacy behavior)
    blos_map_reshaped = blos_map.reshape(nt, ny, nx)
    btrans_map_reshaped = btrans_map.reshape(nt, ny, nx)
    azi_map_reshaped = azi_map.reshape(nt, ny, nx)
    plot_wfa_results(
        blos_map_reshaped[-1],
        btrans_map_reshaped[-1],
        azi_map_reshaped[-1],
        save_name="ref_neural_time_results_last_frame.png",
    )

# 2. Total Loss
loss_val = problem.compute_loss(final_field).item()
print(f"Total Loss: {loss_val:.4e}")

# 3. Baseline WFA Comparison (using PixelSolver for last frame only)
print("Computing Baseline WFA (for comparison)...")
from neural_wfa.optimization import PixelSolver

solver_wfa = PixelSolver(problem, device=device)
solver_wfa.initialize_parameters(method="weak_field")
wfa_field = solver_wfa.get_field()

wfa_blos = torch2numpy(wfa_field.blos_map)
wfa_btrans = torch2numpy(wfa_field.btrans_map)
wfa_azi = torch2numpy(wfa_field.phi_map)

if len(wfa_blos.shape) == 3:  # (Nt, Ny, Nx)
    plot_wfa_results(
        wfa_blos[-1],
        wfa_btrans[-1],
        wfa_azi[-1],
        save_name="ref_neural_time_wfa_baseline.png",
    )
else:
    wfa_blos_r = wfa_blos.reshape(nt, ny, nx)
    wfa_btrans_r = wfa_btrans.reshape(nt, ny, nx)
    wfa_azi_r = wfa_azi.reshape(nt, ny, nx)
    plot_wfa_results(
        wfa_blos_r[-1],
        wfa_btrans_r[-1],
        wfa_azi_r[-1],
        save_name="ref_neural_time_wfa_baseline.png",
    )

## 6. Temporal Evolution at a Single Pixel

In [ ]:
from neural_wfa.utils.viz import plot_temporal_evolution

py, px = 100, 100

# --- NEW: Use smart get_full_field for efficient single-pixel query ---
# Instead of extracting from full grid, query directly!
print(f"Querying pixel (y={py}, x={px}) across all time frames...")

# Neural Field: Query only this pixel (much faster for large grids!)
pixel_field = solver.get_full_field(y=py, x=px)  # Returns (Nt,) shaped field
blos_neural_ts = torch2numpy(pixel_field.blos_map)
btrans_neural_ts = torch2numpy(pixel_field.btrans_map)
azi_neural_ts = torch2numpy(pixel_field.phi_map)

print(f"  Neural Field pixel shape: {blos_neural_ts.shape} (expected: ({nt},))")

# WFA: Also use selective query (PixelSolver doesn't have this yet, so extract from full)
# Reshape maps to (Nt, Ny, Nx) if needed
if len(wfa_blos.shape) != 3:
    wfa_blos = wfa_blos.reshape(nt, ny, nx)
    wfa_btrans = wfa_btrans.reshape(nt, ny, nx)
    wfa_azi = wfa_azi.reshape(nt, ny, nx)

blos_wfa_ts = wfa_blos[:, py, px]
btrans_wfa_ts = wfa_btrans[:, py, px]
azi_wfa_ts = wfa_azi[:, py, px]

# Extract time series at pixel (py, px)
time_frames = np.arange(nt)

# Plot comparison
plot_temporal_evolution(
    time_frames,
    blos_wfa_ts,
    btrans_wfa_ts,
    azi_wfa_ts,  # Method 1: WFA
    blos_neural_ts,
    btrans_neural_ts,
    azi_neural_ts,  # Method 2: Neural
    label_1="WFA (Analytical)",
    label_2="Neural Field",
    pixel_coords=(py, px),
    save_name="ref_neural_time_pixel_evolution.png",
)

print("Saved temporal evolution plot to ref_neural_time_pixel_evolution.png")

# --- Demonstrate single frame query ---
print("\nQuerying single frame (t=2) for full spatial map...")
frame_field = solver.get_full_field(t=2)
print(
    f"  Single frame shape: blos={torch2numpy(frame_field.blos_map).shape} (expected: ({ny}, {nx}))"
)

print("Done. Saved results to ref_neural_time_*.png")